# Time series processing

In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

## Imports

In [ ]:
import os
from pprint import pprint

import xarray as xr

In [ ]:
from sweat.api import read_ts_input_data, run_timeseries
from sweat.timeseries.config import check_config_timeseries
from sweat.timeseries.constant import TimeSeriesVar as TSvar

In [ ]:
import logging

from sweat.logging import LoggerManager

LoggerManager.set_level(logging.INFO)

In [ ]:
def plot_timeseries(data: xr.Dataset, x: int, y: int):
    """
    Plot time series
    """
    data[TSvar.ET.value].isel(x=x, y=y).plot()


def locate_test_data() -> str:
    """
    Locate test data directory
    """
    current = os.getcwd()
    parent = os.path.dirname(current)
    if os.path.isdir(os.path.join(parent, "tests")):
        return os.path.join(parent, "tests", "data", "timeseries")
    parent = os.path.dirname(parent)
    if os.path.isdir(os.path.join(parent, "tests")):
        return os.path.join(parent, "tests", "data", "timeseries")
    msg = "Test data directory not found"
    raise FileNotFoundError(msg)

## Case: Simple execution

Input configuration

In [ ]:
config_dict = {
    "input": {
        "dates": [
            "2025-08-23",
            "2025-08-24",
            "2025-08-25",
            "2025-08-26",
            "2025-08-27",
            "2025-08-28",
            "2025-08-29",
        ],
        "et_time_series": [
            os.path.join(locate_test_data(), "et_time_series_20250823.tif"),
            os.path.join(locate_test_data(), "et_time_series_20250824.tif"),
            os.path.join(locate_test_data(), "et_time_series_20250825.tif"),
            os.path.join(locate_test_data(), "et_time_series_20250826.tif"),
            os.path.join(locate_test_data(), "et_time_series_20250827.tif"),
            os.path.join(locate_test_data(), "et_time_series_20250828.tif"),
        ],
        "radiation": [
            os.path.join(locate_test_data(), "radiation_20250823.tif"),
            os.path.join(locate_test_data(), "radiation_20250824.tif"),
            os.path.join(locate_test_data(), "radiation_20250825.tif"),
            os.path.join(locate_test_data(), "radiation_20250826.tif"),
            os.path.join(locate_test_data(), "radiation_20250827.tif"),
            os.path.join(locate_test_data(), "radiation_20250828.tif"),
            os.path.join(locate_test_data(), "radiation_20250829.tif"),
        ],
        "et_single_date": [
            os.path.join(locate_test_data(), "et_single_date_20250824.tif"),
            os.path.join(locate_test_data(), "et_single_date_20250826.tif"),
            os.path.join(locate_test_data(), "et_single_date_20250828.tif"),
            os.path.join(locate_test_data(), "et_single_date_20250829.tif"),
        ],
    },
    "output": {"path": "out"},
    "params": {},
}

#### Execution

Check input configuration

In [ ]:
config = check_config_timeseries(config_dict)

In [ ]:
pprint(config, sort_dicts=False)

Read

In [ ]:
et_ts, radiation_ts, et_sd, dem = read_ts_input_data(config_dict["input"])

Run 

In [ ]:
updated_ts = run_timeseries(et_ts, radiation_ts, et_sd, dem, config["params"])

In [ ]:
updated_ts

Visualize results

In [ ]:
plot_timeseries(updated_ts, x=0, y=0)

## Case: Detailed configuration

#### Configuration

In [ ]:
config_dict = {
    "input": {
        "dates": [
            "2025-08-23",
            "2025-08-24",
            "2025-08-25",
            "2025-08-26",
            "2025-08-27",
            "2025-08-28",
            "2025-08-29",
        ],
        "et_time_series": [
            os.path.join(locate_test_data(), "et_time_series_20250823.tif"),
            os.path.join(locate_test_data(), "et_time_series_20250824.tif"),
            os.path.join(locate_test_data(), "et_time_series_20250825.tif"),
            os.path.join(locate_test_data(), "et_time_series_20250826.tif"),
            os.path.join(locate_test_data(), "et_time_series_20250827.tif"),
            os.path.join(locate_test_data(), "et_time_series_20250828.tif"),
        ],
        "radiation": [
            os.path.join(locate_test_data(), "radiation_20250823.tif"),
            os.path.join(locate_test_data(), "radiation_20250824.tif"),
            os.path.join(locate_test_data(), "radiation_20250825.tif"),
            os.path.join(locate_test_data(), "radiation_20250826.tif"),
            os.path.join(locate_test_data(), "radiation_20250827.tif"),
            os.path.join(locate_test_data(), "radiation_20250828.tif"),
            os.path.join(locate_test_data(), "radiation_20250829.tif"),
        ],
        "et_single_date": [
            os.path.join(locate_test_data(), "et_single_date_20250824.tif"),
            os.path.join(locate_test_data(), "et_single_date_20250826.tif"),
            os.path.join(locate_test_data(), "et_single_date_20250828.tif"),
            os.path.join(locate_test_data(), "et_single_date_20250829.tif"),
        ],
        "dem": os.path.join(locate_test_data(), "dem.tif"),
    },
    "output": {"path": "out"},
    "params": {
        "stack": {"et_single_date_filtering": {}},
        "update": {
            "method": "linear",
            "params": {"strict_mode": True, "radiation_mode": 0},
        },
    },
}

In [ ]:
pprint(config_dict, sort_dicts=False)

#### Execution

Check input file

In [ ]:
config = check_config_timeseries(config_dict)

Read

In [ ]:
et_ts, radiation_ts, et_sd, dem = read_ts_input_data(config_dict["input"])

Run

In [ ]:
updated_ts = run_timeseries(et_ts, radiation_ts, et_sd, dem, config["params"])

Visualize results

In [ ]:
plot_timeseries(updated_ts, 0, 0)

## Case: detailed pipeline

Additional imports

In [ ]:
from sweat.timeseries import stack_handler as sth
from sweat.timeseries import updater_handler as uh
from sweat.timeseries.config import TimeSeriesParamsConfig

Configuration

In [ ]:
config_dict = {
    "input": {
        "dates": [
            "2025-08-23",
            "2025-08-24",
            "2025-08-25",
            "2025-08-26",
            "2025-08-27",
            "2025-08-28",
            "2025-08-29",
        ],
        "et_time_series": [
            os.path.join(locate_test_data(), "et_time_series_20250823.tif"),
            os.path.join(locate_test_data(), "et_time_series_20250824.tif"),
            os.path.join(locate_test_data(), "et_time_series_20250825.tif"),
            os.path.join(locate_test_data(), "et_time_series_20250826.tif"),
            os.path.join(locate_test_data(), "et_time_series_20250827.tif"),
            os.path.join(locate_test_data(), "et_time_series_20250828.tif"),
        ],
        "radiation": [
            os.path.join(locate_test_data(), "radiation_20250823.tif"),
            os.path.join(locate_test_data(), "radiation_20250824.tif"),
            os.path.join(locate_test_data(), "radiation_20250825.tif"),
            os.path.join(locate_test_data(), "radiation_20250826.tif"),
            os.path.join(locate_test_data(), "radiation_20250827.tif"),
            os.path.join(locate_test_data(), "radiation_20250828.tif"),
            os.path.join(locate_test_data(), "radiation_20250829.tif"),
        ],
        "et_single_date": [
            os.path.join(locate_test_data(), "et_single_date_20250824.tif"),
            os.path.join(locate_test_data(), "et_single_date_20250826.tif"),
            os.path.join(locate_test_data(), "et_single_date_20250828.tif"),
            os.path.join(locate_test_data(), "et_single_date_20250829.tif"),
        ],
        "dem": os.path.join(locate_test_data(), "dem.tif"),
    },
    "output": {"path": "out"},
    "params": {
        "stack": {"et_single_date_filtering": {}},
        "update": {
            "method": "linear",
            "params": {"strict_mode": True, "radiation_mode": 0},
        },
    },
}

Check configuration

In [ ]:
config = check_config_timeseries(config_dict)
# Validate parameters config
params_config = TimeSeriesParamsConfig.model_validate(config["params"])

In [ ]:
pprint(config)

Read input data

In [ ]:
et_ts, radiation_ts, et_sd, dem = read_ts_input_data(config["input"])

In [ ]:
et_ts

In [ ]:
plot_timeseries(et_ts, 0, 0)

Prepare data: filter and stack

In [ ]:
ts, feed = sth.run(
    et_ts,
    radiation_ts,
    et_sd,
    dem,
    et_single_date_filtering=params_config.stack.et_single_date_filtering.model_dump(),
)

In [ ]:
plot_timeseries(ts, 0, 0)

Update time series

In [ ]:
updated_ts = uh.run(ts, feed, params_config.update.model_dump())

In [ ]:
updated_ts

In [ ]:
plot_timeseries(updated_ts, 0, 0)